In [11]:
import math
import numpy as np

from fractions import Fraction

Базовые операции с матрицами


In [12]:
'''Перевод значений в Fraction'''
def to_fraction_matrix(A):
    return np.array([[Fraction(x) for x in row] for row in A], dtype=object)

In [13]:
'''Создание единичной матрицы'''
def identity_matrix(n):
    E = np.zeros((n, n), dtype=object)

    for i in range(n):
        E[i][i] = Fraction(1)

    return E

In [14]:
'''Возведение матрицы в степень'''
def matrix_power(A, p):

    n = A.shape[0]
    result = identity_matrix(n)

    for _ in range(p):
        result = result @ A

    return result

In [15]:
'''Печатанье матрицы'''
def print_matrix(A, name="Матрица"):
    print(f"\n{name}:")

    for row in A:
        out = []

        for x in row:
            if isinstance(x, Fraction):
                if x.denominator == 1:
                    out.append(int(x))
                else:
                    out.append(float(x))
            else:
                out.append(x)

        print(out)

Метод Гаусса

In [16]:
def rref(A):
    A = A.copy()

    n, m = A.shape

    pivot_cols = []

    row = 0

    for col in range(m):

        pivot = None

        for r in range(row, n):
            if A[r][col] != 0:
                pivot = r
                break

        if pivot is None:
            continue

        '''обмен строк'''
        A[[row, pivot]] = A[[pivot, row]]

        '''приведение к 1'''
        pivot_value = A[row][col]
        A[row] = [x / pivot_value for x in A[row]]

        '''вычитания столбца из всех строк'''
        for r in range(n):
            if r != row and A[r][col] != 0:

                factor = A[r][col]

                A[r] = [
                    A[r][c] - factor * A[row][c]
                    for c in range(m)
                ]

        pivot_cols.append(col)

        row += 1

        if row == n:
            break

    return A, pivot_cols

Подсчёт ранга

In [17]:
def rank(A):
    matrix, pivots = rref(A)
    return len(pivots)

Нахождение ядра

In [18]:
def nullspace(A):
    R, pivot_cols = rref(A)

    n = A.shape[1]

    free_cols = [j for j in range(n) if j not in pivot_cols]

    basis = []

    for free in free_cols:

        vec = [Fraction(0) for _ in range(n)]

        vec[free] = Fraction(1)

        for i, pivot_col in enumerate(pivot_cols):
            vec[pivot_col] = -R[i][free]

        basis.append(vec)

    return basis

Подсчёт следов для метода Фаддеева-Леверье

In [19]:
def trace(A):
    n = A.shape[0]

    s = Fraction(0)

    for i in range(n):
        s += A[i][i]

    return s

Характеристический многочлен, метод Фаддеева-Леверье

In [20]:
def characteristic_polynomial(A):
    n = A.shape[0]

    I = identity_matrix(n)

    B = identity_matrix(n)

    coeffs = [Fraction(1)]

    for k in range(1, n + 1):

        AB = A @ B

        c = -trace(AB) / k

        coeffs.append(c)

        B = AB + c * I

    return coeffs

Подстановка в многочлен чисел

In [21]:
def polynomial_value(coeffs, x):
    result = Fraction(0)

    n = len(coeffs)

    for i in range(n):
        result += coeffs[i] * (x ** (n - i - 1))

    return result

Подбор делителей

In [22]:
def divisors(n):
    n = abs(n)

    if n == 0:
        return [0]

    result = set()

    for d in range(1, int(math.sqrt(n)) + 1):

        if n % d == 0:
            result.add(d)
            result.add(-d)
            result.add(n // d)
            result.add(-(n // d))

    return sorted(result)

Собственные значения

In [23]:
def integer_eigenvalues(A):

    coeffs = characteristic_polynomial(A)

    constant = coeffs[-1]

    if constant.denominator != 1:
        raise ValueError("Свободный член не целый")

    roots = []

    for d in divisors(int(constant)):

        if polynomial_value(coeffs, Fraction(d)) == 0:
            roots.append(Fraction(d))

    if polynomial_value(coeffs, Fraction(0)) == 0:
        roots.append(Fraction(0))

    roots = sorted(list(set(roots)))

    multiplicities = {}

    poly = coeffs[:]

    for root in roots:

        count = 0

        while len(poly) > 1:

            quotient = [poly[0]]

            for i in range(1, len(poly) - 1):
                quotient.append(
                    poly[i] + quotient[-1] * root
                )

            remainder = poly[-1] + quotient[-1] * root

            if remainder == 0:

                count += 1

                poly = quotient

            else:
                break

        if count > 0:
            multiplicities[int(root)] = count

    if len(poly) == 2:
        root = -poly[1] / poly[0]

        if root.denominator == 1:
            multiplicities[int(root)] = (
                multiplicities.get(int(root), 0) + 1
            )

    return multiplicities

Подсчёт обратной матрицы

In [24]:
def inverse_matrix(A):
    n = A.shape[0]

    I = identity_matrix(n)

    aug = np.hstack((A.copy(), I))

    R, pivots = rref(aug)

    if len(pivots) < n:
        raise ValueError("Матрица необратима")

    return R[:, n:]

Вспомогательные функции

In [25]:
'''вектор в матрицу'''
def vectors_to_matrix(vectors):
    return np.array(vectors, dtype=object).T


'''матрица минус lambda'''
def matrix_minus_lambda_I(A, lam):
    n = A.shape[0]

    return A - Fraction(lam) * identity_matrix(n)


'''умножение матрицы на вектор'''
def mat_vec_mul(A, v):
    v = np.array(v, dtype=object)

    result = A @ v

    return list(result)


'''проверка что базис не поменялся'''
def in_span(v, basis):

    if not basis:
        return False

    M1 = vectors_to_matrix(basis)

    r1 = rank(M1)

    M2 = vectors_to_matrix(basis + [v])

    r2 = rank(M2)

    return r1 == r2

Подсчёт жордановых цепочек

In [26]:
def jordan_chains(A, lam, alg_mult):

    B = matrix_minus_lambda_I(A, lam)

    kernels = []
    dims = []

    for k in range(1, alg_mult + 1):

        Bk = matrix_power(B, k)

        ker = nullspace(Bk)

        kernels.append(ker)

        dims.append(len(ker))

        if len(ker) == alg_mult:
            break

    chains = []
    used = []

    max_power = len(kernels)

    for level in range(max_power, 0, -1):

        current_kernel = kernels[level - 1]

        lower_kernel = []

        if level > 1:
            lower_kernel = kernels[level - 2]

        needed = len(current_kernel) - len(lower_kernel)

        added = 0

        for v in current_kernel:

            if level > 1 and in_span(v, lower_kernel):
                continue

            chain = []

            cur = v[:]

            for _ in range(level):

                chain.append(cur[:])

                cur = mat_vec_mul(B, cur)

            chain.reverse()

            ok = True

            temp = used[:]

            for vec in chain:

                if in_span(vec, temp):
                    ok = False
                    break

                temp.append(vec)

            if ok:

                chains.append(chain)

                for vec in chain:
                    used.append(vec)

                added += 1

            if added == needed:
                break

    return chains, dims


Жорданова форма

In [27]:
def jordan_form(A):

    A = to_fraction_matrix(A)

    n = A.shape[0]

    eigenvalues = integer_eigenvalues(A)

    print("\nСобственные значения:")

    for lam, mult in eigenvalues.items():
        print(f"lambda = {lam}, алгебраическая кратность = {mult}")

    basis_vectors = []

    jordan_blocks = []

    for lam, mult in eigenvalues.items():

        print(f"\n lambda = {lam} ")

        chains, dims = jordan_chains(A, lam, mult)

        for k, d in enumerate(dims, start=1):
            print(f"dim ker((A-lambda I)^{k}) = {d}")

        for chain in chains:

            size = len(chain)

            jordan_blocks.append((lam, size))

            for v in chain:
                basis_vectors.append(v)

    if len(basis_vectors) != n:
        raise ValueError(
            f"Неполный базис: {len(basis_vectors)} вместо {n}"
        )

    P = vectors_to_matrix(basis_vectors)

    J = np.zeros((n, n), dtype=object)

    idx = 0

    for lam, size in jordan_blocks:

        for i in range(size):

            J[idx + i][idx + i] = Fraction(lam)

            if i < size - 1:
                J[idx + i][idx + i + 1] = Fraction(1)

        idx += size

    return J, P



A = PJP^{-1}

In [28]:
def verify_decomposition(A, J, P):

    A = to_fraction_matrix(A)

    Pinv = inverse_matrix(P)

    reconstructed = P @ J @ Pinv

    print_matrix(reconstructed, "PJP^{-1}")

    ok = True

    n = A.shape[0]

    for i in range(n):
        for j in range(n):

            if reconstructed[i][j] != A[i][j]:
                ok = False

    print(ok)

Тесты


In [29]:
def demo():

    tests = []

    tests.append((
        "Диагонализуемая",
        [
            [4, 1, 0],
            [1, 4, 0],
            [0, 0, 2]
        ]
    ))


    tests.append((
        "Одна жорданова клетка",
        [
            [4, -1, 1],
            [2, 1, 1],
            [0, 0, 2]
        ]
    ))


    tests.append((
        "Нильпотентная",
        [
            [1, 1, -1],
            [1, 1, -1],
            [1, 1, -1]
        ]
    ))


    tests.append((
        "Две клетки",
        [
            [5, 2, 0, 0],
            [0, 5, 0, 0],
            [0, 0, 5, 3],
            [0, 0, 0, 5]
        ]
    ))


    tests.append((
        "Смешанный случай",
        [
            [6, 2, 0, 0, 0],
            [0, 6, 1, 0, 0],
            [0, 0, 6, 0, 0],
            [0, 2, 0, 3, 1],
            [1, 0, 0, 0, 3]
        ]
    ))

    for name, A in tests:

        print("\n" + "=" * 80)
        print(name)
        print("=" * 80)

        print_matrix(to_fraction_matrix(A), "A")

        J, P = jordan_form(A)

        print_matrix(J, "Jordan form J")

        print_matrix(P, "Transition matrix P")

        verify_decomposition(A, J, P)

In [30]:
demo()


Диагонализуемая

A:
[4, 1, 0]
[1, 4, 0]
[0, 0, 2]

Собственные значения:
lambda = 2, алгебраическая кратность = 1
lambda = 3, алгебраическая кратность = 1
lambda = 5, алгебраическая кратность = 1

 lambda = 2 
dim ker((A-lambda I)^1) = 1

 lambda = 3 
dim ker((A-lambda I)^1) = 1

 lambda = 5 
dim ker((A-lambda I)^1) = 1

Jordan form J:
[2, 0, 0]
[0, 3, 0]
[0, 0, 5]

Transition matrix P:
[0, -1, 1]
[0, 1, 1]
[1, 0, 0]

PJP^{-1}:
[4, 1, 0]
[1, 4, 0]
[0, 0, 2]
True

Одна жорданова клетка

A:
[4, -1, 1]
[2, 1, 1]
[0, 0, 2]

Собственные значения:
lambda = 2, алгебраическая кратность = 2
lambda = 3, алгебраическая кратность = 1

 lambda = 2 
dim ker((A-lambda I)^1) = 2

 lambda = 3 
dim ker((A-lambda I)^1) = 1

Jordan form J:
[2, 0, 0]
[0, 2, 0]
[0, 0, 3]

Transition matrix P:
[0.5, -0.5, 1]
[1, 0, 1]
[0, 1, 0]

PJP^{-1}:
[4, -1, 1]
[2, 1, 1]
[0, 0, 2]
True

Нильпотентная

A:
[1, 1, -1]
[1, 1, -1]
[1, 1, -1]

Собственные значения:
lambda = 0, алгебраическая кратность = 2
lambda = 1, алгебра